[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/benchmarks/Benchmark_Classification.ipynb)

# Benchmark: Text Classification

Scores a pretrained document classifier's accuracy against gold-labeled data, using
`sparknlp.benchmark.Benchmark.evaluate(..., task="classification")`.

**Dataset**: [aclImdb](https://ai.stanford.edu/~amaas/data/sentiment/) movie review sentiment.

**Model**: `DistilBertForSequenceClassification.pretrained()` (default:
`distilbert_base_sequence_classifier_imdb`), fine-tuned specifically on this dataset.

## Setup

Run these cells first on a fresh Colab runtime.

In [3]:
!wget https://setup.johnsnowlabs.com/colab.sh -O - | bash

--2026-08-29 10:57:08--  https://setup.johnsnowlabs.com/colab.sh
Resolving setup.johnsnowlabs.com (setup.johnsnowlabs.com)... 3.86.22.73
Connecting to setup.johnsnowlabs.com (setup.johnsnowlabs.com)|3.86.22.73|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh [following]
--2026-08-29 10:57:08--  https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1483 (1.4K) [text/plain]
Saving to: ‘STDOUT’


-                     0%[                    ]       0  --.-KB/s               
-                   100%[===================>]   1.45K  --.-KB/s    in 0s      



In [4]:
# Current Colab runtimes default to Java 21, which Spark 3.4.x (what the bootstrap above
# installs) isn't compatible with -- Spark's low-level Platform.java reflection breaks on it.
# Switch to Java 17, which Spark 3.4.x does support.
!apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [5]:
# Current Colab runtimes also default to Python 3.13, which removed the deprecated
# `typing.io` submodule -- but Spark 3.4.x's own source still does `from typing.io import
# BinaryIO`. This patches that one import, both for this notebook process and for the
# separate Python worker subprocesses Spark launches to actually run distributed tasks
# (those load pyspark from its own bundled zip, so both copies need patching).
import zipfile, shutil

site_pkgs = "/usr/local/lib/python3.13/dist-packages"
loose_path = f"{site_pkgs}/pyspark/broadcast.py"
zip_path = f"{site_pkgs}/pyspark/python/lib/pyspark.zip"
OLD = "from typing.io import BinaryIO  # type: ignore[import]"
NEW = "from typing import BinaryIO  # patched for Python 3.13 (typing.io removed)"

with open(loose_path) as f:
    text = f.read()
with open(loose_path, "w") as f:
    f.write(text.replace(OLD, NEW))

tmp_path = zip_path + ".tmp"
with zipfile.ZipFile(zip_path, "r") as zin, zipfile.ZipFile(tmp_path, "w", zipfile.ZIP_DEFLATED) as zout:
    for item in zin.infolist():
        data = zin.read(item.filename)
        if item.filename == "pyspark/broadcast.py":
            data = data.decode("utf-8").replace(OLD, NEW).encode("utf-8")
        zout.writestr(item, data)
shutil.move(tmp_path, zip_path)
print("Environment patched for this Colab runtime (Java 17, typing.io).")

Environment patched for this Colab runtime (Java 17, typing.io).

In [6]:
import sparknlp
spark = sparknlp.start()
print("Spark NLP version:", sparknlp.version())
print("Apache Spark version:", spark.version)
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import Tokenizer, DistilBertForSequenceClassification
from pyspark.ml import Pipeline
import pyspark.sql.functions as F
from sparknlp.benchmark import Benchmark

Spark NLP version: 6.4.2
Apache Spark version: 3.4.4

## 1. Get some data

aclImdb is small enough that we download and parse a slice of the raw archive directly.

In [8]:
import tarfile
import urllib.request
import io
import random

url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
archive = tarfile.open(fileobj=io.BytesIO(urllib.request.urlopen(url, timeout=120).read()))

rows = []
random.seed(0)
members = [m for m in archive.getmembers()
           if m.name.startswith(("aclImdb/test/pos/", "aclImdb/test/neg/")) and m.name.endswith(".txt")]
random.shuffle(members)
for member in members[:200]:
    label = 1 if "/pos/" in member.name else 0
    text = archive.extractfile(member).read().decode("utf-8")
    rows.append((text, label))

print(len(rows), "reviews")
print(sum(1 for _, l in rows if l == 1), "positive,", sum(1 for _, l in rows if l == 0), "negative")

200 reviews
102 positive, 98 negative

In [9]:
gold_data = spark.createDataFrame(rows, ["text", "label"])

## 2. Build the pipeline

In [11]:
document_assembler = DocumentAssembler().setInputCol("text").setOutputCol("document")
tokenizer = Tokenizer().setInputCols(["document"]).setOutputCol("token")
classifier = DistilBertForSequenceClassification.pretrained() \
    .setInputCols(["document", "token"]).setOutputCol("class")

pipeline = Pipeline(stages=[document_assembler, tokenizer, classifier])
pipeline_model = pipeline.fit(gold_data)

distilbert_base_sequence_classifier_imdb download started this may take some time.
Approximate size to download 234.9 MB

[ | ]
[ / ]
[ — ]
[ \ ]
[ | ]
[ / ]
[ — ]
[ \ ]
[ | ]
[ / ]
[ — ]
[ \ ]
[OK!]

> **Note: check the model's actual label strings before scoring.** `aclImdb`'s own labels
> are the integers `0`/`1`, but the model outputs its own string label names -- a silent
> mismatch here would make every prediction look wrong regardless of how good the model
> actually is. A quick check on a few rows confirms this model outputs lowercase `neg`/`pos`
> (not, say, `NEGATIVE`/`POSITIVE`), which is what we map to below.

In [13]:
label_map = {0: "neg", 1: "pos"}
label_map_udf = F.udf(lambda l: label_map[l], "string")
gold_data = gold_data.withColumn("gold_label", label_map_udf("label"))

## 3. Run the benchmark

In [15]:
report = Benchmark.evaluate(
    pipeline_model, gold_data, task="classification", label_col="gold_label")
print(report)

classification accuracy (n=200): accuracy=0.9500, weightedF1=0.9500, weightedPrecision=0.9508, weightedRecall=0.9500
  neg: f1=0.9500, precision=0.9314, recall=0.9694
  pos: f1=0.9500, precision=0.9694, recall=0.9314